# 04 Event Study

This notebook estimates dynamic event-study specifications and produces publication-style figures.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "final_dataset_ai_exposure_controls.csv")

df["post"] = (df["year"] >= 2023).astype(int)

df = df.rename(columns={
    "log_Median wage": "log_median_wage",
    "log_Median": "log_median_wage",
    "Median wage": "median_wage"
})

df["log_bestand"] = pd.to_numeric(df["log_bestand"], errors="coerce")
if "log_median_wage" in df.columns:
    df["log_median_wage"] = pd.to_numeric(df["log_median_wage"], errors="coerce")

df_controls = df.dropna(subset=[
    "log_relation", "log_vakanz", "ai_exposure",
    "log_bestand", "log_median_wage", "kldb3", "year"
]).copy()

print(df_controls.shape)

## 1. Estimate event-study models

In [ ]:
event_relation = pf.feols(
    "log_relation ~ i(year, ai_exposure, ref=2022) + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

event_vakanz = pf.feols(
    "log_vakanz ~ i(year, ai_exposure, ref=2022) + log_bestand + log_median_wage | kldb3 + year",
    data=df_controls,
    vcov={"CRV1": "kldb3"}
)

print(event_relation.summary())
print(event_vakanz.summary())

## 2. Helper function for event-study coefficients

In [ ]:
def extract_event_study(model, ref_year=2022):
    tidy = model.tidy().reset_index()

    # PyFixest coefficient labels usually contain both year and ai_exposure.
    event_df = tidy[tidy["Coefficient"].str.contains("ai_exposure", na=False)].copy()

    event_df["year"] = (
        event_df["Coefficient"]
        .str.extract(r"(\d{4})")
        .astype(int)
    )

    event_df = event_df.rename(columns={
        "Estimate": "estimate",
        "Std. Error": "se"
    })

    event_df["ci_low"] = event_df["estimate"] - 1.96 * event_df["se"]
    event_df["ci_high"] = event_df["estimate"] + 1.96 * event_df["se"]

    ref = pd.DataFrame({
        "Coefficient": ["Reference"],
        "estimate": [0.0],
        "se": [0.0],
        "year": [ref_year],
        "ci_low": [0.0],
        "ci_high": [0.0]
    })

    event_df = pd.concat([event_df, ref], ignore_index=True)
    event_df = event_df.sort_values("year").reset_index(drop=True)

    return event_df[["year", "estimate", "se", "ci_low", "ci_high", "Coefficient"]]

In [ ]:
event_relation_df = extract_event_study(event_relation)
event_vakanz_df = extract_event_study(event_vakanz)

event_relation_df.to_csv(TABLES / "event_study_log_relation.csv", index=False)
event_vakanz_df.to_csv(TABLES / "event_study_log_vakanz.csv", index=False)

event_relation_df.head()

## 3. Publication-style event-study figures

In [ ]:
def plot_event_study(event_df, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.errorbar(
        event_df["year"],
        event_df["estimate"],
        yerr=[
            event_df["estimate"] - event_df["ci_low"],
            event_df["ci_high"] - event_df["estimate"]
        ],
        fmt="o-",
        capsize=4,
        linewidth=1.8,
        markersize=5
    )

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.axvline(2022, linestyle=":", linewidth=1)

    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle=":", linewidth=0.7)

    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_event_study(
    event_relation_df,
    title="Dynamic Effects of AI Exposure on Labour Market Slack",
    ylabel="Coefficient on AI exposure relative to 2022",
    filename="event_study_log_relation.png"
)

In [ ]:
plot_event_study(
    event_vakanz_df,
    title="Dynamic Effects of AI Exposure on Vacancy Duration",
    ylabel="Coefficient on AI exposure relative to 2022",
    filename="event_study_log_vakanz.png"
)

## 4. Optional combined figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)

for ax, event_df, title in [
    (axes[0], event_relation_df, "Labour market slack"),
    (axes[1], event_vakanz_df, "Vacancy duration")
]:
    ax.errorbar(
        event_df["year"],
        event_df["estimate"],
        yerr=[
            event_df["estimate"] - event_df["ci_low"],
            event_df["ci_high"] - event_df["estimate"]
        ],
        fmt="o-",
        capsize=4,
        linewidth=1.6,
        markersize=4
    )

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.axvline(2022, linestyle=":", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle=":", linewidth=0.7)

axes[0].set_ylabel("Coefficient on AI exposure relative to 2022")

fig.suptitle("Event-Study Estimates", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES / "event_study_combined.png", dpi=300, bbox_inches="tight")
plt.show()